# Практикум 9. Распознавание именованных сущностей: BIO-разметка и модель CRF

В лаборатории 9 вы кликали по токенам, расставляя BIO-метки, включали и выключали признаки и смотрели, как CRF исправляет недопустимые последовательности. Здесь то же самое в коде:

1. переведём размеченные фрагменты текста в BIO-метки и обратно, научимся находить недопустимые последовательности;
2. опишем каждый токен набором признаков (форма слова, соседи, морфология, словарь `pymorphy3`);
3. обучим CRF из `sklearn-crfsuite` на 45 небольших предложениях и посмотрим на выученные веса переходов;
4. сами раскодируем CRF алгоритмом Витерби и посчитаем $P(y \mid x)$, чтобы убедиться, что понимаем, как модель устроена;
5. напишем функцию точности, полноты и F1 на уровне сущностей и сравним наборы признаков.

*Практикум к лаборатории 9 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

In [ ]:
!pip install -q sklearn-crfsuite pymorphy3

## 1. BIO-разметка

Именованная сущность — это фрагмент из одного или нескольких слов: «Лев Толстой» (PER, персона), «Ясная Поляна» (LOC, место), «Казанский университет» (ORG, организация). Чтобы свести поиск фрагментов к разметке последовательности (как в практикуме 8), каждому токену дают метку: **B-X** — первый токен сущности типа X, **I-X** — продолжение, **O** — вне сущности.

Корпус ниже написан для этого практикума: 45 обучающих и 18 тестовых предложений, сущности выделены как `[текст](ТИП)`. Первое предложение — пример из лаборатории.

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

TRAIN_TEXT = """
[Лев Толстой](PER) родился в [Ясной Поляне](LOC) и учился в [Казанском университете](ORG)
[Александр Пушкин](PER) провёл детство в [Москве](LOC)
[Пушкин](PER) учился в [Царскосельском лицее](ORG)
[Анна Петрова](PER) работает в компании [Яндекс](ORG)
Офис [Яндекса](ORG) находится в [Москве](LOC)
[Иван Соколов](PER) переехал из [Казани](LOC) в [Новосибирск](LOC)
Летом мы ездили в [Самарканд](LOC) и [Бухару](LOC)
[Мария](PER) получила премию [Российской академии наук](ORG)
[Сбербанк](ORG) открыл новое отделение в [Ташкенте](LOC)
[Дмитрий Менделеев](PER) родился в [Тобольске](LOC)
Студенты [МГУ](ORG) выступили на конференции в [Санкт-Петербурге](LOC)
[Ольга Иванова](PER) преподаёт физику в [Новосибирском университете](ORG)
Река [Волга](LOC) впадает в [Каспийское море](LOC)
[Газпром](ORG) построил новый офис в [Санкт-Петербурге](LOC)
Вчера [Павел](PER) и [Елена](PER) гуляли по [Ташкенту](LOC)
[Большой театр](ORG) показал новый балет
[Андрей Смирнов](PER) работает врачом в [Казани](LOC)
Наша команда сыграла с клубом [Пахтакор](ORG) в [Ташкенте](LOC)
[Алишер Навои](PER) жил и работал в [Герате](LOC)
В [Самарканде](LOC) сохранилась обсерватория [Улугбека](PER)
[Нина Кузнецова](PER) окончила [Московский университет](ORG) в прошлом году
Поезд из [Москвы](LOC) прибыл в [Нижний Новгород](LOC) утром
[Министерство образования](ORG) объявило новый конкурс
Компания [Аэрофлот](ORG) открыла рейс из [Москвы](LOC) в [Самарканд](LOC)
[Сергей](PER) давно мечтал увидеть [Байкал](LOC)
[Виктор Орлов](PER) возглавил отдел в банке [ВТБ](ORG)
Озеро [Байкал](LOC) находится в [Сибири](LOC)
[Толстой](PER) часто гулял в лесу возле дома
Он читал книгу в толстой обложке
В [Новосибирске](LOC) зимой очень холодно
Учёные [Казанского университета](ORG) открыли новую лабораторию
[Екатерина Белова](PER) выступила в [Большом театре](ORG)
[Наталья](PER) переехала в [Екатеринбург](LOC) к сестре
Банк [ВТБ](ORG) и компания [Яндекс](ORG) подписали соглашение
Мы долго ждали автобус на вокзале
[Михаил Ломоносов](PER) основал [Московский университет](ORG)
[Борис](PER) работает инженером в [Газпроме](ORG)
Весной в [Бухаре](LOC) проходит фестиваль
Мой брат учится в [Ташкентском университете](ORG)
[Игорь Волков](PER) родился в [Нижнем Новгороде](LOC)
Туристы из [Казани](LOC) посетили [Хиву](LOC)
[Алексей](PER) написал письмо в [Министерство образования](ORG)
В понедельник в городе шёл сильный дождь
Летом [Ирина Морозова](PER) отдыхала на [Байкале](LOC)
[Российская академия наук](ORG) наградила молодых учёных
"""
TEST_TEXT = """
[Антон Павлов](PER) работает в [Сбербанке](ORG) в [Новосибирске](LOC)
Делегация из [Ташкента](LOC) посетила [Москву](LOC)
[Елена Соколова](PER) окончила [Казанский университет](ORG)
Сотрудники [Газпрома](ORG) приехали в [Самарканд](LOC)
[Мария](PER) рассказала о поездке на [Волгу](LOC)
[Дмитрий](PER) давно работает в [Яндексе](ORG)
Весной [Павел Орлов](PER) переехал в [Бухару](LOC)
Выпускники [Московского университета](ORG) собрались в [Санкт-Петербурге](LOC)
[Пушкин](PER) часто бывал в [Москве](LOC)
[Ольга](PER) купила билет до [Казани](LOC)
Компания [Аэрофлот](ORG) отменила рейс в [Екатеринбург](LOC)
Бабушка испекла пирог с толстой коркой
[Игорь](PER) и [Анна](PER) поехали на [Байкал](LOC)
[Российская академия наук](ORG) открыла филиал в [Ташкенте](LOC)
Студент из [Хивы](LOC) поступил в [Новосибирский университет](ORG)
Толстая тетрадь лежала на столе
[Дилноза Рахимова](PER) приехала из [Андижана](LOC) в [Наманган](LOC)
[Шахзод](PER) устроился программистом в [Узбектелеком](ORG)
"""

def parse_line(line):
    # разбор строки: [текст](ТИП) — сущность, остальное — отдельные слова
    tokens, spans = [], []
    for m in re.finditer(r"\[([^\]]+)\]\((PER|LOC|ORG)\)|(\S+)", line):
        if m.group(3):
            tokens.append(m.group(3))
        else:
            words = m.group(1).split()
            spans.append((len(tokens), len(tokens) + len(words), m.group(2)))
            tokens += words
    return tokens, spans

def spans_to_bio(n, spans):
    # первое слово сущности — B-ТИП, следующие — I-ТИП, всё остальное — O
    tags = ["O"] * n
    for start, end, typ in spans:
        tags[start] = "B-" + typ
        for i in range(start + 1, end):
            tags[i] = "I-" + typ
    return tags

def read(text):
    out = []
    for line in text.strip().splitlines():
        tokens, spans = parse_line(line)
        out.append((tokens, spans_to_bio(len(tokens), spans)))
    return out

train, test = read(TRAIN_TEXT), read(TEST_TEXT)
for name, data in [("обучение", train), ("тест", test)]:
    n_ent = Counter(t[2:] for _, tags in data for t in tags if t.startswith("B-"))
    print(f"{name}: {len(data)} предложений, {sum(len(t) for t, _ in data)} токенов, "
          f"сущностей: {dict(n_ent)}")

tokens, tags = train[0]
print()
print(pd.DataFrame([tokens, tags], index=["token", "BIO"]).to_string(header=False))

Обратное преобразование нужно, чтобы из предсказанных меток получить сущности. Не всякая последовательность меток допустима: I-X может стоять только после B-X или I-X. Переходы O → I-LOC или B-PER → I-LOC — ошибка. Модель, которая выбирает метку для каждого токена отдельно, может такие ошибки делать.

In [ ]:
def bio_to_spans(tags):
    # I-X, который не продолжает сущность X, начинает новую сущность (как в conlleval)
    spans, start, typ = [], None, None
    for i, t in enumerate(list(tags) + ["O"]):
        continues = t.startswith("I-") and t[2:] == typ
        if start is not None and not continues:
            spans.append((start, i, typ))
            start, typ = None, None
        if t.startswith("B-") or (t.startswith("I-") and not continues):
            start, typ = i, t[2:]
    return spans

def bio_errors(tags):
    # допустимо: I-X только после B-X или I-X
    errs = []
    for i, t in enumerate(tags):
        prev = tags[i - 1] if i > 0 else "O"
        if t.startswith("I-") and prev[2:] != t[2:]:
            errs.append(f"{i}: {prev} → {t}")
    return errs

examples = [
    tags,
    ["B-PER", "I-PER", "O", "O", "O", "I-LOC", "O", "O", "O", "B-ORG", "I-ORG"],
    ["B-PER", "I-LOC", "O", "O", "B-LOC", "I-LOC", "O", "O", "O", "B-ORG", "I-ORG"],
]
for ex in examples:
    errs = bio_errors(ex)
    print(" ".join(ex))
    print("   ", "корректно" if not errs else "ошибки: " + "; ".join(errs))

print("\nСущности, восстановленные из BIO:")
for start, end, typ in bio_to_spans(tags):
    print(f"   {typ}: {' '.join(tokens[start:end])}  [{start}, {end})")

Сущности в русском тексте стоят в разных падежах: «в Ясной Поляне», «из Москвы», «Казанского университета». Для поиска это не мешает, но чтобы, например, сложить все упоминания одного города, сущность нужно привести к начальной форме. Наивная лемматизация по словам с этим справляется плохо:

In [ ]:
import pymorphy3
morph = pymorphy3.MorphAnalyzer()

# наивная нормализация: лемматизируем каждое слово сущности отдельно
for toks, tg in train[:3]:
    for start, end, typ in bio_to_spans(tg):
        words = toks[start:end]
        lemmas = [morph.parse(w)[0].normal_form for w in words]
        print(f"{typ}: {' '.join(words):<24} → {' '.join(lemmas)}")

«Ясной Поляне» превратилось в «ясный полянин»: `pymorphy3` разобрал «Поляне» как форму слова «полянин», а прилагательное привёл к мужскому роду. Нормализация сущностей — отдельная задача: нужно согласовать слова между собой и знать собственные имена. Дальше мы оставляем токены как есть, а лемму используем только как признак.

## 2. Признаки токена

CRF не видит слов, он видит признаки. Для каждого токена соберём словарь `{признак: значение}`. Строковые значения превращаются в индикаторы (`word:москве`), логические — в 0/1. Группы признаков:

- **слово**: само слово в нижнем регистре;
- **форма**: начинается ли с заглавной, всё ли заглавными (МГУ), первое ли в предложении (там заглавная буква ничего не значит);
- **контекст**: соседние слова и их регистр («в» перед словом часто указывает на LOC, «компания» — на ORG);
- **морфология и словарь**: окончания из 2–3 букв, лемма и словарные пометы `pymorphy3`: Name (имя), Surn (фамилия), Geox (топоним), Orgn (организация). Это готовый русский ресурс, по сути — большой словарь имён (газеттир).

In [ ]:
GRAMMEMES = ["Name", "Surn", "Geox", "Orgn"]
_lex_cache = {}

def lex_features(word):
    # pymorphy3: лемма и словарные пометы Name (имя), Surn (фамилия), Geox (топоним), Orgn (организация)
    if word not in _lex_cache:
        parses = morph.parse(word)
        f = {"lemma": parses[0].normal_form}
        for g in GRAMMEMES:
            if any(g in p.tag for p in parses[:3]):
                f["dict:" + g] = True
        _lex_cache[word] = f
    return _lex_cache[word]

ALL_GROUPS = ("word", "shape", "context", "morph")   # группы признаков: слово, форма (заглавная буква), контекст (соседи), морфология + словарь

def token_features(tokens, i, groups=ALL_GROUPS):
    w = tokens[i]
    lw = w.lower()
    f = {"bias": 1.0}
    if "word" in groups:
        f["word"] = lw
    if "shape" in groups:
        f["is_title"] = w[0].isupper()
        f["is_upper"] = w.isupper() and len(w) > 1
        f["is_first"] = i == 0
    if "context" in groups:
        f["-1:word"] = tokens[i - 1].lower() if i > 0 else "<s>"
        f["+1:word"] = tokens[i + 1].lower() if i < len(tokens) - 1 else "</s>"
        f["-1:title"] = i > 0 and tokens[i - 1][0].isupper()
        f["+1:title"] = i < len(tokens) - 1 and tokens[i + 1][0].isupper()
    if "morph" in groups:
        f["suf2"] = lw[-2:]
        f["suf3"] = lw[-3:]
        f.update(lex_features(w))
    return f

def sent_features(tokens, groups=ALL_GROUPS):
    return [token_features(tokens, i, groups) for i in range(len(tokens))]

for i in [1, 4, 9]:
    print(f"Признаки токена «{tokens[i]}»:")
    print("   ", {k: v for k, v in token_features(tokens, i).items() if v is not False})

## 3. Обучение CRF

Линейная CRF (conditional random field) оценивает всю последовательность меток сразу:
$$\text{score}(x, y) = \sum_{i} \sum_{k} w_k\, f_k(x, i, y_i) + \sum_{i} T[y_{i-1}, y_i], \qquad P(y \mid x) = \frac{\exp \text{score}(x, y)}{Z(x)}.$$
Первая сумма — **эмиссии**: веса признаков токена для каждой метки. Вторая — **переходы**: вес пары соседних меток. $Z(x)$ — сумма $\exp \text{score}$ по всем возможным разметкам. В отличие от HMM, модель не порождает слова, а сразу моделирует $P(y \mid x)$, поэтому признаки могут быть любыми и пересекаться. Веса подбираются максимизацией $\log P(y \mid x)$ на обучающих данных с регуляризацией.

In [ ]:
import sklearn_crfsuite

X_train = [sent_features(t) for t, _ in train]
y_train = [g for _, g in train]
X_test = [sent_features(t) for t, _ in test]
y_test = [g for _, g in test]

# c1, c2 — L1- и L2-регуляризация; all_possible_transitions — веса для всех пар меток, даже не встреченных
crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.05, c2=0.05, max_iterations=200,
                           all_possible_transitions=True)
crf.fit(X_train, y_train)
y_pred = [list(p) for p in crf.predict(X_test)]
print(f"{len(crf.state_features_)} признаков с ненулевым весом\n")

print("Предсказания на тесте (в скобках — правильная метка):")
for (toks, gold), pred in zip(test, y_pred):
    print("   ", " ".join(f"{w}/{p}" + ("" if p == g else f"({g})") for w, p, g in zip(toks, pred, gold)))

Выученные веса переходов. Положительный вес — пара меток поощряется, отрицательный — штрафуется. Сравните строку O: переходы O → I-PER, O → I-LOC и O → I-ORG получили отрицательные веса. Модель сама выучила, что I- не может идти после O, хотя мы ей этого не говорили. Переходы B-PER → I-PER и B-ORG → I-ORG, наоборот, сильно положительные.

Ниже — самые сильные признаки-эмиссии: словарные пометы `pymorphy3`, лемма «университет» как продолжение организации, предлог «в» перед местом. Заглавная буква следующего слова (`+1:title`) и первое место в предложении голосуют за O.

In [ ]:
import matplotlib.pyplot as plt

LABELS = ["O", "B-PER", "I-PER", "B-LOC", "I-LOC", "B-ORG", "I-ORG"]
TR = np.array([[crf.transition_features_.get((a, b), 0.0) for b in LABELS] for a in LABELS])
print("Веса переходов (строка — предыдущая метка, столбец — следующая):")
print(pd.DataFrame(TR, index=LABELS, columns=LABELS).round(2).to_string())

fig, ax = plt.subplots(figsize=(6.5, 5))
lim = np.abs(TR).max()
im = ax.imshow(TR, cmap="RdBu", vmin=-lim, vmax=lim)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        ax.text(j, i, f"{TR[i, j]:.1f}", ha="center", va="center", fontsize=8)
ax.set_xticks(range(len(LABELS)), LABELS, rotation=45)
ax.set_yticks(range(len(LABELS)), LABELS)
ax.set_xlabel("следующая метка")
ax.set_ylabel("предыдущая метка")
ax.set_title("Веса переходов CRF")
plt.colorbar(im, fraction=0.04)
plt.tight_layout()
plt.show()

print("Самые сильные признаки-эмиссии:")
for (feat, label), w in Counter(crf.state_features_).most_common(12):
    print(f"   {w:6.2f}  {label:<6} {feat}")

## 4. Как CRF принимает решение: Витерби и $P(y \mid x)$

Проверим, что понимаем модель. Возьмём веса из `crf.state_features_` и `crf.transition_features_`, сами посчитаем эмиссии каждого токена и раскодируем предложение алгоритмом Витерби из практикума 8. Результат должен совпасть с `crf.predict`.

Затем выбросим переходы и выберем лучшую метку для каждого токена отдельно (как независимый классификатор). Для каждого такого предложения напечатаем обе разметки и их сумму: эмиссии + переходы.

In [ ]:
W = crf.state_features_

def emission_matrix(feats):
    # сумма весов всех признаков токена для каждой метки (эмиссии)
    E = np.zeros((len(feats), len(LABELS)))
    for i, f in enumerate(feats):
        for key, val in f.items():
            name, x = (key + ":" + val, 1.0) if isinstance(val, str) else (key, float(val))
            if x:
                for j, lab in enumerate(LABELS):
                    E[i, j] += W.get((name, lab), 0.0) * x
    return E

def viterbi(E, T):
    # Витерби: тот же, что в практикуме 8, только вместо логарифмов вероятностей — веса
    n = len(E)
    delta, back = E[0].copy(), []
    for i in range(1, n):
        scores = delta[:, None] + T
        back.append(scores.argmax(axis=0))
        delta = scores.max(axis=0) + E[i]
    path = [int(delta.argmax())]
    for b in reversed(back):
        path.append(int(b[path[-1]]))
    return [LABELS[j] for j in reversed(path)]

def score(E, T, tags):
    idx = [LABELS.index(t) for t in tags]
    em = sum(E[i, j] for i, j in enumerate(idx))
    tr = sum(T[a, b] for a, b in zip(idx, idx[1:]))
    return em, tr

same, invalid = 0, []
for (toks, gold), feats, pred in zip(test, X_test, y_pred):
    E = emission_matrix(feats)
    same += viterbi(E, TR) == pred
    indep = [LABELS[j] for j in E.argmax(axis=1)]     # = viterbi(E, 0 * TR)
    if bio_errors(indep):
        invalid.append((toks, E, indep, pred))
print(f"предложений теста, где наш Витерби совпал с crf.predict: {same}/{len(test)}")
print(f"предложений теста, где без переходов получается некорректная BIO-разметка: {len(invalid)}\n")

for toks, E, indep, pred in invalid:
    print(" ".join(toks))
    for name, tg in [("Только эмиссии (каждый токен отдельно):", indep), ("CRF (эмиссии + переходы):", pred)]:
        em, tr = score(E, TR, tg)
        bad = "  ← некорректная BIO: " + "; ".join(bio_errors(tg)) if bio_errors(tg) else ""
        print(f"   {name:<38} {' '.join(tg)}")
        print(f"   {'':<38} эмиссии {em:6.2f} + переходы {tr:6.2f} = сумма {em + tr:6.2f}{bad}")
    print()

Без переходов модель местами ставит I- после O или после метки другого типа. «Казанский» и «Московского» получили I-PER: в словаре `pymorphy3` есть фамилии Казанский и Московский (помета Surn), а целостность фрагмента никто не проверяет. «Дилнозы» в словаре нет, и без переходов имя теряется, остаётся «висящая» фамилия I-PER. CRF отдаёт за эмиссии немного баллов, но выигрывает на переходах.

Теперь вероятность. $Z(x)$ считается алгоритмом прямого хода: это Витерби, где максимум заменён на сумму (в логарифмах — `logaddexp`). Для короткого предложения можно перебрать все $7^n$ разметок и проверить, что $P(y \mid x)$ совпадает с тем, что считает сама crfsuite, а сумма по всем разметкам равна 1.

In [ ]:
import itertools

def log_partition(E, T):
    # прямой проход: как Витерби, но вместо max — log-sum-exp; получаем log Z(x)
    alpha = E[0].copy()
    for i in range(1, len(E)):
        alpha = np.logaddexp.reduce(alpha[:, None] + T, axis=0) + E[i]
    return np.logaddexp.reduce(alpha)

toks, gold = test[15]
feats = X_test[15]
E = emission_matrix(feats)
logZ = log_partition(E, TR)
tagger = crf.tagger_
tagger.set(feats)
print(" ".join(toks))
print(f"log Z(x) = {logZ:.3f}   ({len(LABELS)}^{len(toks)} = {len(LABELS) ** len(toks)} разметок)\n")

cands = []
for tg in itertools.product(LABELS, repeat=len(toks)):
    em, tr = score(E, TR, tg)
    cands.append((em + tr, tg))
cands.sort(reverse=True)
print("Самые вероятные разметки (вероятности по всем путям суммируются в 1):")
for s, tg in cands[:5]:
    p_ours = np.exp(s - logZ)
    print(f"   {' '.join(tg):<34} score {s:6.2f}   P = {p_ours:.4f}   crfsuite: {tagger.probability(list(tg)):.4f}")
print(f"   Σ P = {np.exp(np.logaddexp.reduce([s for s, _ in cands]) - logZ):.4f}")

## 5. Качество: точность, полнота и F1 по сущностям

Метки O составляют большую часть токенов, поэтому доля верных меток по токенам выглядит хорошо даже у слабой модели. Для NER считают качество по сущностям: сущность найдена верно, только если совпали её начало, конец и тип.
$$P = \frac{TP}{TP + FP}, \qquad R = \frac{TP}{TP + FN}, \qquad F_1 = \frac{2PR}{P + R}.$$
Здесь TP — найденные верно сущности, FP — лишние (нет в эталоне), FN — пропущенные. Подробнее о метриках — в практикуме 18.

In [ ]:
def entity_prf(gold_seqs, pred_seqs):
    # строгое совпадение: начало, конец и тип сущности
    counts = defaultdict(lambda: [0, 0, 0])      # tp, fp, fn
    for g, p in zip(gold_seqs, pred_seqs):
        G, P = set(bio_to_spans(g)), set(bio_to_spans(p))
        for s in P:
            counts[s[2]][0 if s in G else 1] += 1
        for s in G - P:
            counts[s[2]][2] += 1
    rows = {}
    for typ in sorted(counts) + ["все"]:
        tp, fp, fn = [sum(counts[t][k] for t in counts) for k in range(3)] if typ == "все" else counts[typ]
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        rows[typ] = dict(tp=tp, fp=fp, fn=fn, P=p, R=r, F1=f)
    return pd.DataFrame(rows).T

from sklearn_crfsuite import metrics
report = entity_prf(y_test, y_pred)
print(report.round(3).to_string())
print(f"\nдоля верных меток по токенам (вместе с O): {metrics.flat_accuracy_score(y_test, y_pred):.3f}")
print(f"F1 по сущностям: {report.loc['все', 'F1']:.3f}")

print("\nОшибки на уровне сущностей:")
for (toks, gold), pred in zip(test, y_pred):
    G, P = set(bio_to_spans(gold)), set(bio_to_spans(pred))
    for s, e, t in sorted(G - P):
        print(f"   пропущена: {t:<4} {' '.join(toks[s:e])}")
    for s, e, t in sorted(P - G):
        print(f"   лишняя: {t:<4} {' '.join(toks[s:e])}")

Разберите ошибки. «Толстая тетрадь» — заглавная буква в начале предложения и помета Surn у слова «толстая» (есть фамилия Толстая) сделали из тетради персону (в разделе 4 видно, что модель колебалась: 0,42 против 0,31). Имена из словаря `pymorphy3` модель находит, даже если не видела их в обучении: «Шахзод» (помета Name), «Узбектелеком» (Orgn), «Наманган» (Geox). «Дилнозы» в словаре нет, её спасает переход B-PER → I-PER к фамилии «Рахимова».

Какие признаки важнее? Обучим CRF с постепенно расширяющимся набором групп признаков. На 18 тестовых предложениях одна сущность — это около 3% F1, так что небольшие колебания — шум. Большой скачок даёт группа «морфология»: словарные пометы `pymorphy3` работают как огромный список имён и названий.

In [ ]:
settings = [
    ("слово", ("word",)),
    ("+ форма", ("word", "shape")),
    ("+ контекст", ("word", "shape", "context")),
    ("+ морфология", ALL_GROUPS),
]
f1s = []
for name, groups in settings:
    m = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.05, c2=0.05, max_iterations=200,
                             all_possible_transitions=True)
    m.fit([sent_features(t, groups) for t, _ in train], y_train)
    pred = [list(p) for p in m.predict([sent_features(t, groups) for t, _ in test])]
    rep = entity_prf(y_test, pred)
    f1s.append(rep.loc["все", "F1"])
    print(f"{name:<16} P = {rep.loc['все', 'P']:.3f}   R = {rep.loc['все', 'R']:.3f}   F1 = {f1s[-1]:.3f}")

plt.figure(figsize=(6, 3.2))
plt.bar([n for n, _ in settings], f1s, color=["#9AA9C9", "#7088C0", "#4C6CC4", "#2B55C9"])
for i, v in enumerate(f1s):
    plt.text(i, v + 0.01, f"{v:.2f}", ha="center")
plt.ylim(0, 1.08)
plt.ylabel("F1")
plt.title("F1 по сущностям на тесте при разных наборах признаков", fontsize=10)
plt.tight_layout()
plt.show()

## 6. Задания

1. **Независимый классификатор.** Обучите `LogisticRegression` на тех же признаках (`DictVectorizer` из `sklearn.feature_extraction` превращает словари в матрицу; каждый токен — отдельный пример). Посчитайте F1 по сущностям функцией `entity_prf` и число тестовых предложений с некорректной BIO (`bio_errors`). Сравните с CRF.
2. **Регуляризация.** Обучите CRF с `c1` = 0, 0.5, 2 (при `c2=0.05`). Как меняется число признаков с ненулевым весом (`len(crf.state_features_)`) и F1? Почему L1 обнуляет веса?
3. **Ошибка «Толстая тетрадь».** Добавьте в `TRAIN_TEXT` 2–3 предложения, где слова «толстый», «толстая» в начале предложения не являются именами, и переобучите модель. Исправилась ли ошибка? Не испортилось ли «Толстой» как PER?
4. **Нестрогое совпадение.** Напишите `entity_prf_partial`, где предсказанная сущность считается верной, если она пересекается хотя бы одним токеном с эталонной сущностью того же типа. Сравните со строгой метрикой на модели без группы «морфология».
5. **Схема BIOES.** Напишите перевод BIO → BIOES (E — последний токен многословной сущности, S — сущность из одного токена) и обратно. Обучите CRF на BIOES, переведите предсказания в BIO и сравните F1.

*Совет: свои предложения в формате `[текст](ТИП)` можно сохранить в `my_ner.txt`, загрузить через панель «Файлы» и прочитать функцией `read(open("my_ner.txt").read())`.*

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь